# 44. Response Filtering: Post-Processing Outputs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/05-output-control/44_response_filtering.ipynb)

**Category:** Output Control and Formatting  **Technique #:** 44  **Difficulty:** Intermediate

## 📋 Description

Response Filtering applies post-processing rules to clean, validate, and refine LLM outputs. This technique ensures outputs meet quality standards, remove unwanted content, and conform to application requirements.

**When to use:**
- Content moderation and safety filtering
- Removing PII (Personally Identifiable Information)
- Normalizing output formats
- Quality assurance checks
- Preparing outputs for downstream systems

## 🔧 How It Works

Response Filtering applies post-processing rules to clean LLM outputs:

1. **Content Safety Check** - Remove inappropriate content
2. **PII Detection and Removal** - Mask personal information
3. **Format Normalization** - Standardize structure
4. **Quality Validation** - Check completeness
5. **Output Cleaning** - Remove artifacts

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install openai -q

import os
from getpass import getpass
from openai import OpenAI
import re
import json

# Set up API key securely
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

def get_response(prompt, model="gpt-4o-mini"):
    """Get raw response from LLM."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

# Define filter functions
class ResponseFilter:
    """Filter pipeline for LLM responses."""
    
    @staticmethod
    def remove_pii(text):
        """Remove common PII patterns."""
        # Email addresses
        text = re.sub(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}', '[EMAIL]', text)
        # Phone numbers
        text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE]', text)
        # SSN
        text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN]', text)
        # Credit cards (simplified)
        text = re.sub(r'\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b', '[CREDIT_CARD]', text)
        return text
    
    @staticmethod
    def extract_json(text):
        """Extract JSON from text."""
        # Find JSON block
        json_match = re.search(r'```json\s*(.*?)\s*```', text, re.DOTALL)
        if json_match:
            return json_match.group(1)
        
        # Find raw JSON
        json_match = re.search(r'\{.*\}', text, re.DOTALL)
        if json_match:
            return json_match.group(0)
        
        return text
    
    @staticmethod
    def remove_thinking(text):
        """Remove thinking/reasoning sections."""
        # Remove content between thinking tags
        text = re.sub(r'<thinking>.*?</thinking>', '', text, flags=re.DOTALL)
        text = re.sub(r'\[Thinking:.*?\]', '', text, flags=re.DOTALL)
        # Remove introductory phrases
        text = re.sub(r'^(Let me|I will|First, let me).*?\n', '', text, flags=re.IGNORECASE)
        return text.strip()
    
    @staticmethod
    def normalize_whitespace(text):
        """Normalize whitespace."""
        text = re.sub(r'\n{3,}', '\n\n', text)  # Max 2 newlines
        text = re.sub(r' {2,}', ' ', text)  # Single spaces
        return text.strip()

## 💡 Basic Example

In [ ]:
# Basic filtering example
raw_response = '''
Let me think about this...

Okay, here is the information:

Contact: John Smith
Email: john.smith@email.com
Phone: 555-123-4567
SSN: 123-45-6789

```json
{
  "name": "John Smith",
  "department": "Engineering"
}
```

Hope this helps!
'''

print("Raw Response:")
print("=" * 50)
print(raw_response)

# Apply filters
filter_pipeline = [
    ("Remove thinking", ResponseFilter.remove_thinking),
    ("Remove PII", ResponseFilter.remove_pii),
    ("Extract JSON", ResponseFilter.extract_json),
    ("Normalize whitespace", ResponseFilter.normalize_whitespace)
]

print("\n" + "=" * 50)
print("Filtering Pipeline:")
print("=" * 50)

current = raw_response
for name, filter_fn in filter_pipeline:
    current = filter_fn(current)
    print(f"\n{name}:")
    print("-" * 30)
    print(current[:200] + "..." if len(current) > 200 else current)

## 🌍 Real-World Example: Customer Data Sanitization

In [ ]:
# Real-world: Sanitize customer service responses
def sanitize_for_sharing(raw_text, allowed_fields=None):
    """Sanitize text for external sharing."""
    if allowed_fields is None:
        allowed_fields = ['status', 'category', 'priority']
    
    # Step 1: Remove all PII
    sanitized = ResponseFilter.remove_pii(raw_text)
    
    # Step 2: Remove internal notes
    sanitized = re.sub(r'\[Internal:.*?\]', '', sanitized, flags=re.DOTALL)
    
    # Step 3: Remove agent signatures
    sanitized = re.sub(r'--\s*\n.*$', '', sanitized, flags=re.DOTALL)
    
    # Step 4: Normalize
    sanitized = ResponseFilter.normalize_whitespace(sanitized)
    
    return sanitized

# Sample customer service response with PII
customer_response = '''
Dear Mr. Johnson,

Thank you for contacting our support team.

[Internal: Customer is VIP, handle with priority]

We have reviewed your case #12345 regarding the billing issue.
Our records show your account (johnson.m@company.com) was charged
incorrectly on 10/15/2024.

We have processed a refund of $149.99 to your card ending in 4567.
The refund will appear in 3-5 business days.

If you need further assistance, please call us at 1-800-555-0199.

Best regards,
--
Sarah Williams
Senior Support Agent
Direct: 555-987-6543
'''

print("Customer Service Response - Before Sanitization:")
print("=" * 50)
print(customer_response)

print("\n" + "=" * 50)
print("After Sanitization:")
print("=" * 50)
sanitized = sanitize_for_sharing(customer_response)
print(sanitized)

print("\n" + "=" * 50)
print("Sanitization Summary:")
print("  ✓ Email addresses masked")
print("  ✓ Phone numbers masked")
print("  ✓ Credit card info masked")
print("  ✓ Internal notes removed")
print("  ✓ Agent signatures removed")

## ❌ Failure Case: Insufficient Filtering

In [ ]:
# Failure case: Missing filters
print("BAD EXAMPLE - Insufficient Filtering:")
print("=" * 50)

raw_output = '''
Here is the customer information:

Name: Jane Doe
Email: jane.doe@personal.com
Phone: 555-234-5678
Address: 123 Main St, Anytown, USA
SSN: 987-65-4321
Credit Card: 4111-1111-1111-1111

This data is for internal use only.
'''

# No filtering - dangerous!
print("Unfiltered Output (DANGEROUS):")
print(raw_output)
print("\n❌ Problem: All PII exposed - privacy violation!")

print("\n" + "=" * 50)
print("GOOD EXAMPLE - Proper Filtering:")
print("=" * 50)

# Apply comprehensive filtering
filtered = ResponseFilter.remove_pii(raw_output)
filtered = ResponseFilter.normalize_whitespace(filtered)

print("Filtered Output:")
print(filtered)
print("\n✅ Success: PII masked, safe for sharing")

## 📊 Benchmark: Filtering Pipeline Performance

In [ ]:
import time

# Benchmark filtering pipeline
test_outputs = [
    "Contact: john@email.com, Phone: 555-123-4567, SSN: 123-45-6789",
    "Let me think... The answer is 42. Hope this helps!",
    "```json\n{\"name\": \"Test\"}\n```\nExtra text here",
    "Line 1\n\n\n\n\nLine 2    with    spaces"
]

filters = {
    "No filter": lambda x: x,
    "PII only": ResponseFilter.remove_pii,
    "Thinking only": ResponseFilter.remove_thinking,
    "JSON only": ResponseFilter.extract_json,
    "Whitespace only": ResponseFilter.normalize_whitespace,
    "Full pipeline": lambda x: ResponseFilter.normalize_whitespace(
        ResponseFilter.extract_json(
            ResponseFilter.remove_pii(
                ResponseFilter.remove_thinking(x))))
}

print("BENCHMARK: Filtering Pipeline Performance\n")
print(f"{'Filter':<20} {'Avg Time (ms)':<15} {'PII Removed':<12} {'Cleaned'}")
print("-" * 65)

for filter_name, filter_fn in filters.items():
    times = []
    pii_removed = 0
    cleaned = 0
    
    for output in test_outputs:
        start = time.time()
        result = filter_fn(output)
        times.append((time.time() - start) * 1000)  # Convert to ms
        
        # Check if PII was removed
        if '@' not in result and '[EMAIL]' in result:
            pii_removed += 1
        
        # Check if output was cleaned
        if len(result) < len(output) or result != output:
            cleaned += 1
    
    avg_time = sum(times) / len(times)
    pii_status = f"{pii_removed}/4" if pii_removed > 0 else "N/A"
    clean_status = f"{cleaned}/4"
    
    print(f"{filter_name:<20} {avg_time:.3f}         {pii_status:<12} {clean_status}")

print("\nKey Findings:")
print("• All filters execute in <1ms")
print("• Full pipeline provides comprehensive cleaning")
print("• PII filter is essential for privacy compliance")
print("• Combine filters based on use case requirements")

## 🎮 Interactive Playground

In [ ]:
# Interactive filter builder
def create_filter_pipeline(filters):
    """Create a custom filter pipeline."""
    def pipeline(text):
        for name, filter_fn in filters:
            text = filter_fn(text)
        return text
    return pipeline

# Example: Create a production-ready filter
production_filters = [
    ("thinking", ResponseFilter.remove_thinking),
    ("pii", ResponseFilter.remove_pii),
    ("normalize", ResponseFilter.normalize_whitespace)
]

production_pipeline = create_filter_pipeline(production_filters)

# Test with sample LLM outputs
test_outputs = [
    "Let me think... The customer email is support@company.com and phone is 555-999-8888",
    "Here is the data: ```json\n{\"id\": 123}\n```\nLet me know if you need more!",
    "User SSN: 555-55-5555, Card: 4111-1111-1111-1111\n\n\n\nThanks!"
]

print("Production Filter Pipeline")
print("=" * 50)
print(f"Filters: {[f[0] for f in production_filters]}\n")

for i, output in enumerate(test_outputs, 1):
    print(f"\nTest {i}:")
    print("-" * 30)
    print(f"Input: {output[:60]}...")
    filtered = production_pipeline(output)
    print(f"Output: {filtered[:60]}...")

# Try building your own pipeline!
print("\n" + "=" * 50)
print("Try modifying the production_filters list above!")

## 💡 Tips and Tricks

### Filter Design Best Practices

1. **Layer your filters** - Apply in order: content then PII then format
2. **Use regex carefully** - Test patterns thoroughly
3. **Log filter actions** - Track what is being removed
4. **Make filters reversible** - When possible, for debugging
5. **Handle edge cases** - Empty strings, None values
6. **Performance matters** - Filters should be fast (<1ms)

### Common Filter Patterns

```python
# Content filter
def filter_inappropriate(text):
    blocked_words = ['spam', 'inappropriate_word']
    for word in blocked_words:
        text = text.replace(word, '[REMOVED]')
    return text

# Length filter
def truncate_if_needed(text, max_length=1000):
    return text[:max_length] + "..." if len(text) > max_length else text

# Validation filter
def validate_json(text):
    try:
        json.loads(text)
        return text
    except:
        return '{"error": "Invalid JSON"}'
```

### Production Considerations

- **Audit logging** - Log all filter actions for compliance
- **Configurable rules** - Make filters configurable without code changes
- **Metrics** - Track filter hit rates
- **Fail-safe** - Default to safe output on filter errors

## 📚 References

1. [Presidio PII Detection](https://microsoft.github.io/presidio/)
2. [Python Regex Documentation](https://docs.python.org/3/library/re.html)
3. [GDPR Data Protection](https://gdpr.eu/)
4. [Content Moderation Best Practices](https://www.microsoft.com/en-us/security/blog/)